<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "084"
iso_country = "Belize"
year = 2025 

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [4]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Belize': 

Number of trade records present in dataset: 11788
Number of partner countries present in dataset: 99
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1000.2m
Total weight within the dataset: 623.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,084,Belize,724,Spain,Import,Sea,842320,842320 | Weighing machines; s,8423,8423 | Weighing machines; e,84,84 | Machinery and mechan,43922,Weighing machinery (excluding balances of a se...,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,7.115,NaN,7.115,u,0.0,3.280,0.860915,8167.0,5074.736957,4.368917,['Container'],['Container'],0.014477,0.014421,0.014425,0.014425,0.014461,0.014461,0.015216,0.028803,0.016803,0.015477,0.015538,0.015993,0.014865,0.027305,0.017495,0.016104,0.015736,0.016536,0.014557,0.030303,0.022006,0.019137,0.017684,0.017285,0.014254,0.032159,0.025724,0.024018,0.018712,0.018596,0.013956,0.032794,0.030883,0.031431,0.020274,0.019563,0.013652,0.032245,0.030860,0.032669,0.024514,0.019935,0.013367,0.028561,0.029405,0.030703,0.02279,0.020180,0.013076,0.025690,0.026296,0.030241,0.024997,0.020340,43922
1,1,084,Belize,156,China,Import,Sea,401310,"401310 | Rubber; inner tubes,",4013,"4013 | Inner tubes, of rubb",40,40 | Rubber and articles,36114,"Inner tubes, solid or cushion tyres, interchan...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",27347.280,NaN,27347.280,NaN,0.0,3490.252,7427.521248,17859.0,11097.064689,82423.683768,['Container'],['Container'],273.123659,272.057206,272.134491,272.134491,272.822772,272.822772,287.068274,543.394908,317.011787,291.979584,293.137129,301.723445,280.441175,515.130002,330.053070,303.820169,296.879395,311.959008,274.635409,571.692860,415.161848,361.030436,333.629503,326.092725,268.907665,606.716342,485.309258,453.127963,353.012905,350.830250,263.284501,618.695646,582.638425,592.974063,382.491340,369.067147,257.553811,608.332878,582.204126,616.337908,462.486479,376.093546,252.177706,538.832105,554.756639,579.240130,429.94743,380.710708,246.697389,484.666741,496.091667,570.529446,471.599550,383.738177,36114


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $155.3m (15.5%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
22 | Beverages, spirits a  23260399.7  15076094.9      2.3      2.4
21 | Miscellaneous edible  23218903.5   7065483.9      2.3      1.1
15 | Animal, vegetable or  20839863.8  13254115.7      2.1      2.1
04 | Dairy produce; birds  20598783.0   5550831.8      2.1      0.9
19 | Preparations of cere  15609711.8   5537745.5      1.6      0.9
20 | Preparations of vege  10030309.7   7771047.2      1.0      1.2
16 | Meat, fish, crustace   9140209.5   2076145.0      0.9      0.3
10 | Cereals                8894361.6  30897509.3      0.9      5.0
11 | Products of the mill   6223204.6  13088141.3      0.6      2.1
23 | Food industries, res   4983549.4   3693668.7      0.5      0.6
09 | Coffee, tea, mate an   4759316.3   1274529.0      0.5      0.2
02 | Meat and e

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [6]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $203.4m (20.3%), ... 

                     cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Food & Beverages  145406495.8  43533566.9     14.5      7.0
Agriculture        57987581.4  41781922.8      5.8      6.7
Fishing               14281.5      3115.9      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $121.3m (12.1%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     65356880.0  34984806.7      6.5      5.6
MIL - Dairy products           20226257.4   5312652.6      2.0      0.9
VOL - Vegetable oils and fats  19887027.4  12694726.8      2.0      2.0
OMT - Meat products n.e.c.     10620945.1   2396217.6      1.1      0.4
V_F - Vegetables, fruit, nuts   1344926.2   2311090.0      0.1      0.4
OCR - Crops n.e.c.              1279258.6    307286.7      0.1      0.0
GRO - Cereal grains n.e.c.      1002223.6   4143119.6      0.1      0.7
PDR - Paddy rice                 872916.0   1516255.1      0.1      0.2
OSD - Oil seeds                  290711.0    435231.9      0.0      0.1
SGR - Sugar                      160997.5    353445.7      0.0      0.1
PCR - Processed rice             118707.2    206194.5      0.0      0.

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 15.5% 
GTAP 12.1% 
EORA 20.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_14582/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,084,Belize,724,Spain,Import,Sea,842320,842320 | Weighing machines; s,8423,8423 | Weighing machines; e,84,84 | Machinery and mechan,43922,Weighing machinery (excluding balances of a se...,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,7.115,NaN,7.115,u,0.0,3.280,0.860915,8167.0,5074.736957,4.368917,['Container'],['Container'],0.014477,0.014421,0.014425,0.014425,0.014461,0.014461,0.015216,0.028803,0.016803,0.015477,0.015538,0.015993,0.014865,0.027305,0.017495,0.016104,0.015736,0.016536,0.014557,0.030303,0.022006,0.019137,0.017684,0.017285,0.014254,0.032159,0.025724,0.024018,0.018712,0.018596,0.013956,0.032794,0.030883,0.031431,0.020274,0.019563,0.013652,0.032245,0.030860,0.032669,0.024514,0.019935,0.013367,0.028561,0.029405,0.030703,0.02279,0.020180,0.013076,0.025690,0.026296,0.030241,0.024997,0.020340,43922,NaN
1,1,084,Belize,156,China,Import,Sea,401310,"401310 | Rubber; inner tubes,",4013,"4013 | Inner tubes, of rubb",40,40 | Rubber and articles,36114,"Inner tubes, solid or cushion tyres, interchan...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",27347.280,NaN,27347.280,NaN,0.0,3490.252,7427.521248,17859.0,11097.064689,82423.683768,['Container'],['Container'],273.123659,272.057206,272.134491,272.134491,272.822772,272.822772,287.068274,543.394908,317.011787,291.979584,293.137129,301.723445,280.441175,515.130002,330.053070,303.820169,296.879395,311.959008,274.635409,571.692860,415.161848,361.030436,333.629503,326.092725,268.907665,606.716342,485.309258,453.127963,353.012905,350.830250,263.284501,618.695646,582.638425,592.974063,382.491340,369.067147,257.553811,608.332878,582.204126,616.337908,462.486479,376.093546,252.177706,538.832105,554.756639,579.240130,429.94743,380.710708,246.697389,484.666741,496.091667,570.529446,471.599550,383.738177,36114,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $100.5m of $1000.2m (10.0%)
Volume: 81.7 of 623.6 million kilograms (13.1%)
Value-to-Weight: 1.23 from 1.6 US$ per kg (+-23.3%)
Average distance to market (volume-weighted): 2723.9 from 2535.0 nm (+7.4%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    268  42494871.8  26001222.4      4.2      4.2
1        Fats and Oils     57  20702399.0  13031299.3      2.1      2.1
2                Dairy     42  20274956.7   5369724.1      2.0      0.9
3  Unprocessed Cereals     16   7902738.4  29175059.7      0.8      4.7
4           Vegetables    113   7679398.8   7088863.2      0.8      1.1
5      Sugar and Honey     40   1133440.9    915008.3      0.1      0.1
6        Meat and Prep     15    178169.1     70163.7      0.0      0.0
7               Fruits     66    116909.2     47661.5      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [12]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,7914503.6,14976648.7,8739343.6,8049624.8,8081506.1,8318305.7,7259187.8,17052151.6,16058254.5,16343438.9,10544288.9,10173855.2,6802161.6,13359368.8,13674653.9,15726572.4,12999571.3,10578077.3
1,Fats and Oils,7630306.1,13914315.7,8348070.1,7723962.8,7752353.3,7993853.8,7041421.0,15833113.6,14864974.1,15154308.5,10057161.7,9664451.8,6630668.0,12529029.5,12908002.4,14895923.6,12280940.9,10026556.5
2,Fruits,30658.9,57766.3,33817.5,31167.9,31289.6,32210.5,28141.5,65779.4,61941.9,63059.2,40791.8,39339.8,26385.8,51599.9,52834.1,60762.2,50227.3,40892.2
3,Meat and Prep,25646.4,48546.3,28321.5,26085.1,26188.6,26955.7,23521.6,55273.6,52052.3,52975.7,34171.4,32972.1,22039.7,43299.6,44320.3,50970.5,42132.2,34282.8
4,Processed Cereals,24892444.4,46411663.5,27381594.3,25277611.6,25373885.0,26126491.2,22890592.1,52952106.8,49873742.3,50825844.1,33043768.2,31795024.6,21494415.0,41642520.8,42637077.4,48993099.8,40540160.8,33025991.3
5,Sugar and Honey,929617.0,1709674.7,1019200.2,942720.5,946129.4,974541.0,856828.8,1950993.7,1836708.8,1873108.8,1228147.7,1180954.8,806045.5,1540157.7,1578754.4,1814542.8,1501526.0,1225734.8
6,Unprocessed Cereals,24939814.7,41452973.7,26628897.6,25021289.8,25100903.2,25906140.7,23368706.6,48960098.6,46265758.6,47805416.3,32571691.2,30324612.2,22272522.1,39375094.9,40165665.2,45522877.9,38305076.1,31235705.2
7,Vegetables,13122843.8,24771488.7,14481546.8,13343513.3,13395922.6,13789271.2,12041474.2,28207264.5,26563274.9,27039582.3,17470018.2,16851474.8,11287307.2,22114926.8,22639582.4,26035411.8,21522162.9,17518368.5


### 3.3.2 Total FEM costs by scenario (% cif)

In [13]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.35,0.04,0.01,0.01,0.02,0.48,0.43,0.45,0.16,0.14,0.32,0.34,0.44,0.31,0.19
1,Fats and Oils,0.30,0.03,0.00,0.01,0.02,0.42,0.38,0.39,0.15,0.13,0.28,0.30,0.40,0.27,0.16
2,Fruits,0.23,0.03,0.00,0.01,0.01,0.32,0.29,0.30,0.11,0.10,0.22,0.23,0.29,0.20,0.12
3,Meat and Prep,0.13,0.02,0.00,0.00,0.01,0.18,0.16,0.17,0.06,0.05,0.12,0.13,0.16,0.11,0.07
4,Processed Cereals,0.51,0.06,0.01,0.01,0.03,0.71,0.63,0.66,0.24,0.21,0.47,0.50,0.65,0.45,0.27
5,Sugar and Honey,0.69,0.08,0.01,0.01,0.04,0.97,0.86,0.90,0.33,0.29,0.65,0.68,0.89,0.61,0.37
6,Unprocessed Cereals,2.09,0.21,0.01,0.02,0.12,3.24,2.90,3.09,1.16,0.88,2.16,2.26,2.94,2.03,1.13
7,Vegetables,1.52,0.18,0.03,0.04,0.09,2.11,1.89,1.95,0.71,0.63,1.41,1.48,1.92,1.33,0.81
